# Coleta de dados - fluxo instagram login

Base URL: https://graph.instagram.com
Docs: https://developers.facebook.com/docs/instagram-platform/instagram-api-with-instagram-login
Docs: https://developers.facebook.com/docs/instagram-platform/instagram-graph-api/reference/ig-user


# 1. Setup 

In [1]:
import requests
import json
import os
from datetime import datetime
from dotenv import load_dotenv
import sys

sys.path.insert(0, os.path.abspath(".."))

In [2]:
load_dotenv()

GRAPH_VERSION = "v25.0"
BASE_URL = "https://graph.instagram.com"

access_token = os.getenv("LONG_LIVED_TOKEN_IG")
ig_user_id = os.getenv("PROFILE_ID_IG")

# 2. Perfil

In [3]:
fields = "id,username,name,biography,website,followers_count,follows_count,media_count,profile_picture_url,account_type"

# url = f"{BASE_URL}/{GRAPH_VERSION}/{ig_user_id}?fields={fields}&access_token={access_token}"
url = f"{BASE_URL}/{GRAPH_VERSION}/me?fields={fields}&access_token={access_token}"

In [4]:
response = requests.get(url)

In [5]:
if response.status_code == 200:
    profile = response.json()
    print(json.dumps(profile, indent=2, ensure_ascii=False))

else:
    print("Erro: ", response.status_code)
    print(response.text)

{
  "id": "25400762076282923",
  "username": "technews24.7",
  "name": "Notícias Tech 24/7",
  "biography": "noticias tech",
  "followers_count": 2,
  "follows_count": 25,
  "media_count": 2,
  "profile_picture_url": "https://scontent-for2-2.cdninstagram.com/v/t51.82787-19/549498754_17844264411575187_2607132828623270266_n.jpg?stp=dst-jpg_s206x206_tt6&_nc_cat=103&ccb=7-5&_nc_sid=bf7eb4&efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLnd3dy4zMzMuQzMifQ%3D%3D&_nc_ohc=-lpy4Y5E9qAQ7kNvwEOPvjs&_nc_oc=AdlSaprgBdRrD6uBasQb4AKM6i9cgesVRpD5QvlGqCEo4W_sWhPXXt_F8CzGDcU8OTprb9XshMtL4YFspJrvB770&_nc_zt=24&_nc_ht=scontent-for2-2.cdninstagram.com&edm=AP4hL3IEAAAA&_nc_gid=SMIwukz-3ljZmG0wTp2WhQ&_nc_tpa=Q5bMBQE7a06BqDpg_irx1Y7ZWioTEEWB4i37aBrJ-CK7hkXfiKqkA9ylmK0vu2F7EPueq0xoUmFungbq5g&oh=00_AfzVvJSKkKI00ukupglQxgXysnOzN_z8xwwmnal2mqZ9rQ&oe=69BDBB3F",
  "account_type": "MEDIA_CREATOR"
}


# 3. Posts/Media

Primeira página

In [6]:
media_fields = "id,caption,media_type,media_url,thumbnail_url,permalink,timestamp,like_count,comments_count"

url = f"{BASE_URL}/{GRAPH_VERSION}/{ig_user_id}/media?fields={media_fields}&limit=100&access_token={access_token}"


In [7]:
response = requests.get(url)

In [8]:
if response.status_code == 200:
    data = response.json()
    posts = data.get("data",[])
    paging = data.get("paging", {})

    print(f"Posts da página: {len(posts)}")
    print(f"Cursos 'after' disponível: {bool(paging.get('cursos',{}).get('after'))}")
    print("PRIMEIRO post: ")
    print(json.dumps(posts[0],indent=2,ensure_ascii=False))
    print("\nTODO os DADOS:")
    print(json.dumps(data,indent=2,ensure_ascii=False))

else:
    print("erro: ", response.status_code)
    print(response.text)


Posts da página: 2
Cursos 'after' disponível: False
PRIMEIRO post: 
{
  "id": "18406296661186436",
  "caption": "#### ATENÇÃO ####\nO CAFEZINHO PRETO NO COPO AMERICANO É OFICIALMENTE A BEBIDA DOS DEVS",
  "media_type": "IMAGE",
  "media_url": "https://scontent-for2-2.cdninstagram.com/v/t51.82787-15/583465111_17853982278575187_3238836366871602539_n.jpg?stp=dst-jpg_e35_tt6&_nc_cat=108&ccb=7-5&_nc_sid=18de74&efg=eyJlZmdfdGFnIjoiRkVFRC5iZXN0X2ltYWdlX3VybGdlbi5DMyJ9&_nc_ohc=5mgCa9PxebcQ7kNvwHzzEnB&_nc_oc=AdkPzJTt_mFHuX0ChDZJMVphbSmgz2EanU101zAdtf2vNjhRGtDuJflry4BPigQbpAV70qKG4S95TCENgZlHtkm3&_nc_zt=23&_nc_ht=scontent-for2-2.cdninstagram.com&edm=ANo9K5cEAAAA&_nc_gid=wSrCC2WENjAW6gKbCBDvMw&_nc_tpa=Q5bMBQFBpgCT3L92lDUACzfomfguBhgXuxMxSt7RySydHeEVDHWolWBJvYnfFtlC_SILSH6_B5r2PDzi0A&oh=00_Afyy5MyjdgBH2_IY2HBnEHXp2YiwuekRmDzXnYU7ka3F7Q&oe=69BDE823",
  "permalink": "https://www.instagram.com/p/DRKwmmPkfgS/",
  "timestamp": "2025-11-17T18:45:30+0000",
  "like_count": 4,
  "comments_count": 3
}

TODO

Todas as páginas

In [9]:
all_posts = []
next_url = url
page_num = 1

while next_url:
    response = requests.get(next_url)

    if response.status_code != 200:
        print(f"Erro na página: {page_num}: {response.status_code}")
        print(response.text)
        break

    data = response.json()
    posts = data.get("data",[])
    paging = data.get("paging", {})

    all_posts.extend(posts)
    print(f"Pagina {page_num:>3} | +{len(posts):>3} posts | Total: {len(all_posts):>5}")

    next_url = paging.get("next")
    page_num += 1

print(f"\nTotal de posts coletados: {len(all_posts)}")

Pagina   1 | +  2 posts | Total:     2

Total de posts coletados: 2


# 4. Comentários + Replies

Comentários também são paginados, o padrão de cursos de aplica

In [10]:
# Comentários do primeiro post
sample_post_id = all_posts[0]["id"]

comment_fields = "id,text,timestamp,like_count,username"

url_comments = f"{BASE_URL}/{GRAPH_VERSION}/{sample_post_id}/comments?fields={comment_fields}&limit=100&access_token={access_token}"


In [11]:
response = requests.get(url_comments)

In [12]:
if response.status_code == 200:
    data = response.json()
    comments = data.get("data", [])
    print(f"Comentários retornados: {len(comments)}")
    if comments:
        print("\nExemplo do primeiro comentário:")
        print(json.dumps(comments[0], indent=2, ensure_ascii=False))
else:
    print("Erro:", response.status_code)
    print(response.text)

Comentários retornados: 3

Exemplo do primeiro comentário:
{
  "id": "18412438912127005",
  "text": "Top",
  "timestamp": "2026-02-27T13:20:38+0000",
  "like_count": 0
}


Replies do comentário, se existirem

In [13]:
if comments:
    sample_comment_id = comments[0]["id"]

    reply_fields = "id,text,timestamp,username"

    url_replies = (f"{BASE_URL}/{GRAPH_VERSION}/{sample_comment_id}/replies?fields={reply_fields}&limit=100&access_token={access_token}")

    response = requests.get(url_replies)

    if response.status_code == 200:
        replies = response.json().get("data",[])
        print(f"Replies encontradas: {len(replies)}")
        if replies:
            print(json.dumps(replies=[0],indent=2,ensure_ascii=False))
        else:
            print("Nenhuma reply neste comentário.")

    else: 
        print("Erro:", response.status_code)
        print(response.text)
else:
    print("Nenhum comentário disponível para buscar replies.")

Replies encontradas: 0
Nenhuma reply neste comentário.


# 5. Insights de posts

As métricas variam por media_type

- Image/Carrousel: `impressions, reach, saved, shares, total_interactions`
- Video/Reels: `impressions, reach, saved, shares, total_interactions, plays`
- Story: `impressions, reach, exits, replies, taps_forward, taps_back`

`impressions` esta depreciado


Insight de imagem/carrossel

In [16]:
sample_post_id = all_posts[0]["id"]
sample_type = all_posts[0].get("media_type","IMAGE")

print(f"Post {sample_post_id} - tipo: {sample_type}")

metrics_map = {
    "IMAGE": "reach,saved,shares,total_interactions",
    "CAROUSEL_ALBUM": "reach,saved,shares,total_interactions",
    "VIDEO": "reach,saved,shares,total_interactions,plays",
}
metrics = metrics_map.get(sample_type, "reach,saved,shares,total_interactions")

url_insights = f"{BASE_URL}/{GRAPH_VERSION}/{sample_post_id}/insights?metric={metrics}&access_token={access_token}"


Post 18406296661186436 - tipo: IMAGE


In [18]:
response = requests.get(url_insights)


if response.status_code == 200:
    insights = response.json().get("data", [])
    print("Insights do post:")
    print(json.dumps(insights,indent=2,ensure_ascii=False))
else:
    print("Erro:", response.status_code)
    print(response.text)

Insights do post:
[
  {
    "name": "reach",
    "period": "lifetime",
    "values": [
      {
        "value": 2
      }
    ],
    "title": "Contas alcançadas",
    "description": "O número de contas únicas que viram esse post pelo menos uma vez. O alcance é diferente das impressões, que podem incluir várias visualizações do seu post pelas mesmas contas. Essa métrica é estimada.",
    "id": "18406296661186436/insights/reach/lifetime"
  },
  {
    "name": "saved",
    "period": "lifetime",
    "values": [
      {
        "value": 0
      }
    ],
    "title": "Salvo",
    "description": "O número de vezes que seu post foi salvo.",
    "id": "18406296661186436/insights/saved/lifetime"
  },
  {
    "name": "shares",
    "period": "lifetime",
    "values": [
      {
        "value": 0
      }
    ],
    "title": "Compartilhamentos",
    "description": "O número de compartilhamentos do seu post.",
    "id": "18406296661186436/insights/shares/lifetime"
  },
  {
    "name": "total_interacti

Veja que existe o parametro period(que nos ultimos dados estão como lifetime), vale observar se há como alterar isso

Coletar insights de múltiplos posts em loop

In [21]:
posts_com_insights = []
erros = []

#10 primeiros
for post in all_posts[:10]:
    media_id = post['id']
    media_type = post.get("media_type","IMAGE")
    metrics = metrics_map.get(media_type, "reach,saved,shares,total_interactions")

    url_i = f"{BASE_URL}/{GRAPH_VERSION}/{media_id}/insights?metric={metrics}&access_token={access_token}"

    resp = requests.get(url_i)

    if resp.status_code == 200:
        insight_data = {i["name"]: (i["values"][0]["value"] if i.get("values") else i.get("value"))
                        for i in resp.json().get("data", [])}
        posts_com_insights.append({**post, **insight_data})
    else:
        err = resp.json().get("error", {}).get("message", resp.text)
        erros.append({"id": media_id, "erro": err})
        print(f"ERROR: {media_id} | {media_type:<15} | {err}")

print(f"Total de insights: {len(posts_com_insights)}  |  Erros: {len(erros)}")

ERROR: 18091336891776984 | IMAGE           | A mídia foi postada antes do momento mais recente em que a conta do usuário foi convertida de uma conta pessoal para uma conta comercial.
Total de insights: 1  |  Erros: 1


In [26]:
print(posts_com_insights)

[{'id': '18406296661186436', 'caption': '#### ATENÇÃO ####\nO CAFEZINHO PRETO NO COPO AMERICANO É OFICIALMENTE A BEBIDA DOS DEVS', 'media_type': 'IMAGE', 'media_url': 'https://scontent-for2-2.cdninstagram.com/v/t51.82787-15/583465111_17853982278575187_3238836366871602539_n.jpg?stp=dst-jpg_e35_tt6&_nc_cat=108&ccb=7-5&_nc_sid=18de74&efg=eyJlZmdfdGFnIjoiRkVFRC5iZXN0X2ltYWdlX3VybGdlbi5DMyJ9&_nc_ohc=5mgCa9PxebcQ7kNvwHzzEnB&_nc_oc=AdkPzJTt_mFHuX0ChDZJMVphbSmgz2EanU101zAdtf2vNjhRGtDuJflry4BPigQbpAV70qKG4S95TCENgZlHtkm3&_nc_zt=23&_nc_ht=scontent-for2-2.cdninstagram.com&edm=ANo9K5cEAAAA&_nc_gid=SD0QAq3LJckKzVOEq0B32w&_nc_tpa=Q5bMBQHIL_a5BbU4Oy2RyOvYzxB28sl3-yqEtxgcCJHkpl_KVcp2bPf_0SXTdz-3_pOfSqQ8593Ikldl2g&oh=00_AfyttpwKcoFvwnmZEJx2FxucYofH0YfYw2yTxtQzo7lbag&oe=69BDE823', 'permalink': 'https://www.instagram.com/p/DRKwmmPkfgS/', 'timestamp': '2025-11-17T18:45:30+0000', 'like_count': 4, 'comments_count': 3, 'reach': 2, 'saved': 0, 'shares': 0, 'total_interactions': 7}]


# 6. Insights da Conta

- `period=day/week/month` → métricas clássicas (`impressions`, `reach`, `profile_views`)
- `metric_type=total_value` → métricas novas com `since/until` em Unix timestamp

In [38]:
import time

since = int(time.mktime(datetime(2026,2,16).timetuple()))
until = int(time.time())

In [39]:
url_account_insights = f"{BASE_URL}/{GRAPH_VERSION}/{ig_user_id}/insights?metric=reach,profile_views&period=day&since={since}&until={until}&access_token={access_token}"

response = requests.get(url_account_insights)

In [40]:
if response.status_code == 200:
    data = response.json()
    print("Insights da conta (por dia):")
    for metric in data.get("data", []):
        print(f"\n  Métrica : {metric['name']} (period={metric.get('period')})")
        valores = metric.get("values", [])[:5]  # mostra os 5 primeiros dias
        for v in valores:
            print(f"    {v['end_time'][:10]} : {v['value']}")
else:
    print("Erro:", response.status_code)
    print(response.text)

Insights da conta (por dia):

  Métrica : reach (period=day)
    2026-02-16 : 0
    2026-02-17 : 0
    2026-02-18 : 0
    2026-02-19 : 0
    2026-02-20 : 0


In [43]:
url_audience = (
    f"{BASE_URL}/{GRAPH_VERSION}/{ig_user_id}/insights"
    f"?metric=reached_audience_demographics,engaged_audience_demographics,follower_demographics"
    f"&period=lifetime"
    f"&metric_type=total_value"
    f"&breakdown=city,age,gender"
    f"&access_token={access_token}"
)

response = requests.get(url_audience)

if response.status_code == 200:
    data = response.json()
    print("Insights de audiência:")
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    print("Erro:", response.status_code)
    print(response.text)

Insights de audiência:
{
  "data": []
}
